In [ ]:
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import distinctipy
import pickle
from collections import Counter
from tqdm import tqdm 
import cv2
from pathlib import Path
from scipy.ndimage import binary_dilation
from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib
from scipy.spatial import KDTree
from collections import Counter

In [ ]:
!pip install tqdm_joblib

In [ ]:

#given cell_segmentation_mask
#for each cell
#compute # of cells assigned to target phenotype around each cell 

In [ ]:
radius = 100

In [ ]:
key = "" # sample identifier

In [ ]:
cell_clustering_base_path = r"" # path to cell phenotyping results

In [ ]:
pixel_clustering_path = r"" # path to pixel clustering results
cell_pixel_barcode_path = r"" # path to cell pixel barcode results

In [ ]:
pixels_outside_cells = pd.read_csv(cell_pixel_barcode_path + '/pixel_proportions_outside_cells.csv').drop(columns=['Unnamed: 0'])

In [ ]:
with open(cell_clustering_base_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
with open(cell_clustering_base_path + '/cell_type_color_dict.pickle', 'rb') as f: 
    cell_type_color_dict = pickle.load(f)

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_base_path + '/cell_cluster_location_merged_unclassified_within_contour.csv').drop(columns=['Unnamed: 0', 'index', 'index.1'])

In [ ]:
from scipy.ndimage import distance_transform_edt
def compute_cell_count_outside_cell(cell_idx):
    ys, xs = np.where(cell_segmentation_mask == cell_idx)
    y_min = max(ys.min() - radius, 0)
    y_max = min(ys.max() + radius + 1, cell_segmentation_mask.shape[0])
    x_min = max(xs.min() - radius, 0)
    x_max = min(xs.max() + radius + 1, cell_segmentation_mask.shape[1])
    
    cell_block = (cell_segmentation_mask[y_min:y_max, x_min:x_max] == cell_idx)
    mask_block =  cell_segmentation_mask[y_min:y_max, x_min:x_max]
    dist = distance_transform_edt(~cell_block) 
    ring_mask = (dist <= radius) & (~cell_block)
    cell_ids = np.unique(mask_block[ring_mask])
    cell_types =  list(cell_cluster_location_df[cell_cluster_location_df['Cell ID'].isin(cell_ids)]['Cell Type'])
    
    return Counter(cell_types)


In [ ]:
cell_id_list = list(cell_cluster_location_df['Cell ID'])

In [ ]:
with tqdm_joblib(tqdm(desc="Processing cells", total=len(cell_id_list))) as progress_bar:
    cell_counts_around_cells = Parallel(n_jobs=-1)(delayed(compute_cell_count_outside_cell)(idx) for idx in cell_id_list)

In [ ]:
with open(cell_pixel_barcode_path + '/cell_counts_outside_cells.pickle', 'wb') as f: 
    pickle.dump(cell_counts_around_cells, f)